# SWR Detection — Sharp-Wave Ripples in Hippocampal iEEG

Detects ripple-band events (80–200 Hz) from the raw monopolar BIDS EDF (500 Hz).  
Works independently from the existing preprocessing pipeline — does **not** use `_raw.fif`.

**Pipeline:**
1. Load hippocampal channel names from `electrode_locations.csv` (AAL label = `Hippocampus_*`)
2. Load bad channels from Stage 2 `.annot` file — exclude them
3. Load raw monopolar EDF (500 Hz) + notch filter
4. Band-pass 80–200 Hz → Hilbert envelope → z-score → threshold → events
5. Visual inspection of detected events
6. Summary statistics

In [ ]:
# 0. Imports & Config
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, hilbert
from scipy.ndimage import gaussian_filter1d
import mne
mne.set_log_level('WARNING')

# ── project root: notebook lives in mycode/, project root is one level up ──
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
BIDS_ROOT    = os.path.join(PROJECT_ROOT, 'ds004789-download')
PROC_ROOT    = os.path.join(PROJECT_ROOT, 'dr-processed')

# ── subject / session ──────────────────────────────────────────────────────
SUBJECT = 'sub-R1002P'
SESSION = 'ses-1'

# ── detection parameters ───────────────────────────────────────────────────
SFREQ_EXPECTED = 500      # Hz  — raw BIDS EDF
RIPPLE_LO      = 150       # Hz  — bandpass low edge
RIPPLE_HI      = 200      # Hz  — bandpass high edge  (Nyquist = 250 Hz)
Z_THRESHOLD    = 3.0      # SD above mean
MAX_Z          = 20.0     # SD — events above this are likely IED/artifact, not SWR
MIN_DURATION   = 0.015    # s   — minimum event length (~15 ms)
MAX_DURATION   = 0.500    # s   — maximum event length (rejects epileptic spikes)
MERGE_GAP      = 0.020    # s   — merge events separated by less than this
SMOOTH_SIGMA_S = 0.004    # s   — Gaussian smoothing sigma (4 ms)

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'Subject/Session: {SUBJECT} / {SESSION}')

## 1. Hippocampal Channels + Bad Channel Filtering

Two sources of information:
- **`electrode_locations.csv`** — which electrodes are in hippocampus (AAL label)
- **`.annot` file** (Stage 2 output) — which bipolar channels were marked bad by `noise_classifier`

Bad bipolar channels (e.g. `LDA1-LDA2`) are mapped back to their monopolar components.  
Any monopolar electrode appearing in a bad bipolar pair is excluded.

In [ ]:
def get_hippocampal_channels(subject):
    """Return monopolar electrode names labelled Hippocampus_L or Hippocampus_R."""
    csv_path = os.path.join(BIDS_ROOT, subject, f'{subject}_electrode_locations.csv')
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f'Electrode CSV not found: {csv_path}')
    df = pd.read_csv(csv_path)
    hippo = df[df['AAL_Label'].str.contains('Hippocampus', na=False)].copy()
    channels = hippo['name'].tolist()
    print(f'{subject} — hippocampal contacts ({len(channels)} total):')
    for _, row in hippo.iterrows():
        print(f'  {row["name"]:8s}  {row["AAL_Label"]:20s}  MNI=({row["x"]:.1f}, {row["y"]:.1f}, {row["z"]:.1f})')
    return channels


def get_bad_monopolar_channels(subject, session):
    """
    Load bad channels from the Stage 2 .annot file (created for the bipolar montage).
    Maps bad bipolar pairs back to their monopolar components.
    e.g. bad bipolar 'LDA1-LDA2' -> exclude monopolars LDA1 and LDA2.
    """
    annot_path = os.path.join(
        PROC_ROOT, subject, session,
        f'{subject}_{session}_task-FR1_acq-bipolar_ieeg.annot'
    )
    if not os.path.exists(annot_path):
        print(f'  [warn] .annot not found — no bad channels excluded: {annot_path}')
        return []

    with open(annot_path, 'rb') as f:
        annot_data = pickle.load(f)

    bad_bipolar = annot_data.get('bads', [])
    print(f'  Bad bipolar channels in .annot ({len(bad_bipolar)}): {bad_bipolar}')

    # Map bipolar -> monopolar: 'LDA1-LDA2' -> ['LDA1', 'LDA2']
    bad_monopolar = set()
    for ch in bad_bipolar:
        for part in ch.split('-'):
            bad_monopolar.add(part.strip())

    return sorted(bad_monopolar)


# ── Run ────────────────────────────────────────────────────────────────────
hippo_channels = get_hippocampal_channels(SUBJECT)
print()
bad_channels   = get_bad_monopolar_channels(SUBJECT, SESSION)

good_hippo = [ch for ch in hippo_channels if ch not in bad_channels]
bad_hippo  = [ch for ch in hippo_channels if ch in bad_channels]

print(f'\nGood hippocampal channels ({len(good_hippo)}): {good_hippo}')
if bad_hippo:
    print(f'Excluded as bad       ({len(bad_hippo)}): {bad_hippo}')

## 2. Load Raw Monopolar EDF

Source: `ds004789-download/{sub}/{ses}/ieeg/{sub}_{ses}_task-FR1_acq-monopolar_ieeg.edf`  
Sampling rate: 500 Hz (broadband, before any preprocessing).  
Notch filter applied at 60/120/180/240 Hz (powerline harmonics).

In [ ]:
edf_path = os.path.join(
    BIDS_ROOT, SUBJECT, SESSION, 'ieeg',
    f'{SUBJECT}_{SESSION}_task-FR1_acq-monopolar_ieeg.edf'
)
print(f'Loading: {edf_path}')

raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
assert raw.info['sfreq'] == SFREQ_EXPECTED, \
    f'Expected {SFREQ_EXPECTED} Hz, got {raw.info["sfreq"]} Hz'
print(f'  {len(raw.ch_names)} channels  |  {raw.info["sfreq"]:.0f} Hz  |  {raw.times[-1]:.1f} s')

# Keep only good hippocampal channels present in the EDF
available = [ch for ch in good_hippo if ch in raw.ch_names]
not_in_edf = [ch for ch in good_hippo if ch not in raw.ch_names]
if not_in_edf:
    print(f'  [warn] Not found in EDF (name mismatch?): {not_in_edf}')
    print(f'  EDF channel names containing "DA" or "DH": '
          f'{[c for c in raw.ch_names if "DA" in c or "DH" in c]}')

raw.pick_channels(available)
print(f'  Analysing {len(available)} channels: {available}')

# Notch filter: powerline + harmonics
raw.notch_filter(freqs=[60, 120, 180, 240], verbose=False)
print('  Notch filter applied (60 / 120 / 180 / 240 Hz)')

## 3. Ripple Detection

**Algorithm (per channel):**
1. Band-pass Butterworth filter (80–200 Hz, order 4)
2. Hilbert transform → analytic amplitude envelope
3. Gaussian smoothing (σ = 4 ms)
4. Robust z-score (median ± IQR — insensitive to events themselves)
5. Threshold crossings: `Z_THRESHOLD < z < MAX_Z` (upper cutoff removes IEDs)
6. Duration filter: `[MIN_DURATION, MAX_DURATION]`
7. Merge events separated by < `MERGE_GAP`

**`MAX_Z = 20`** — events with peak z > 20 are excluded: at that amplitude they are  
almost certainly **Interictal Epileptic Discharges (IED)**, not physiological SWR.

In [ ]:
def detect_ripples(signal, sfreq,
                   lo=RIPPLE_LO, hi=RIPPLE_HI,
                   z_thresh=Z_THRESHOLD, max_z=MAX_Z,
                   min_dur=MIN_DURATION, max_dur=MAX_DURATION,
                   merge_gap=MERGE_GAP,
                   smooth_sigma_s=SMOOTH_SIGMA_S):
    """
    Detect ripple-band events in a 1-D signal.

    Returns
    -------
    events     : list of (start_sample, end_sample)
    filtered   : band-passed signal
    z_envelope : smoothed, z-scored envelope
    """
    nyq = sfreq / 2

    # 1. Band-pass filter
    b, a = butter(6, [lo / nyq, hi / nyq], btype='band')
    filtered = filtfilt(b, a, signal)

    # 2. Hilbert envelope
    envelope = np.abs(hilbert(filtered))

    # 3. Gaussian smoothing
    sigma_samp = smooth_sigma_s * sfreq
    envelope = gaussian_filter1d(envelope, sigma=sigma_samp)

    # 4. Robust z-score  (median + IQR — unbiased by events)
    med = np.median(envelope)
    iqr = np.percentile(envelope, 75) - np.percentile(envelope, 25)
    z_envelope = (envelope - med) / (iqr / 1.3490)   # IQR/1.349 ≈ SD for Gaussian

    # 5. Threshold crossings
    above = z_envelope > z_thresh
    diff  = np.diff(above.astype(np.int8))
    starts = np.where(diff ==  1)[0] + 1
    ends   = np.where(diff == -1)[0] + 1
    if above[0]:    starts = np.concatenate([[0],            starts])
    if above[-1]:   ends   = np.concatenate([ends, [len(above)]])

    # 6. Duration filter
    min_samp = int(min_dur * sfreq)
    max_samp = int(max_dur * sfreq)
    events = [(int(s), int(e)) for s, e in zip(starts, ends)
              if min_samp <= (e - s) <= max_samp]

    # 7. Merge nearby events
    merge_samp = int(merge_gap * sfreq)
    merged = []
    for s, e in events:
        if merged and (s - merged[-1][1]) < merge_samp:
            merged[-1] = (merged[-1][0], max(e, merged[-1][1]))
        else:
            merged.append([s, e])

    # 8. Remove IED-like events (z > MAX_Z)
    clean = [(s, e) for s, e in merged
             if z_envelope[s:e].max() <= max_z]
    n_ied = len(merged) - len(clean)

    return clean, filtered, z_envelope, n_ied


# ── Run on all good hippocampal channels ───────────────────────────────────
data, times = raw.get_data(return_times=True)
sfreq = raw.info['sfreq']

all_events   = {}   # ch_name -> list of (start_samp, end_samp)
all_filtered = {}   # ch_name -> filtered trace
all_z        = {}   # ch_name -> z-score envelope

print(f'{'Channel':<12} {'Events':>8} {'IED removed':>12}')
print('-' * 36)
for i, ch in enumerate(raw.ch_names):
    evts, filt, z_env, n_ied = detect_ripples(data[i], sfreq)
    all_events[ch]   = evts
    all_filtered[ch] = filt
    all_z[ch]        = z_env
    print(f'{ch:<12} {len(evts):>8}   {n_ied:>8} IED')

# ── Build summary DataFrame ─────────────────────────────────────────────────
rows = []
for ch, evts in all_events.items():
    for s, e in evts:
        peak_idx = s + int(all_z[ch][s:e].argmax())
        rows.append({
            'channel':     ch,
            'start_s':     float(times[s]),
            'end_s':       float(times[e - 1]),
            'duration_ms': float((e - s) / sfreq * 1000),
            'peak_z':      float(all_z[ch][s:e].max()),
            'peak_time_s': float(times[peak_idx]),
        })

df_events = pd.DataFrame(rows)
print(f'\nTotal: {len(df_events)} events across {df_events["channel"].nunique()} channels')
if not df_events.empty:
    display(df_events.describe().round(2))

## 4. Event Inspector

Random sample of detected events — 3-panel plot per event:
- **Top:** raw LFP (unfiltered) — look for the **sharp wave**: a brief, local negative deflection
- **Middle:** 80–200 Hz filtered trace — look for the **ripple burst**: 3–6 fast oscillations
- **Bottom:** z-score envelope + threshold

Red shading = detected event window.

In [ ]:
CONTEXT_S  = 0.200   # s — context window before/after event
N_EXAMPLES = min(12, len(df_events))

if df_events.empty:
    print('No events detected. Try lowering Z_THRESHOLD (e.g. to 2.5).')
else:
    sample = df_events.sample(N_EXAMPLES, random_state=42).sort_values('start_s')

    for _, row in sample.iterrows():
        ch  = row['channel']
        t0  = row['start_s']
        t1  = row['end_s']

        i0 = max(0, int((t0 - CONTEXT_S) * sfreq))
        i1 = min(len(times) - 1, int((t1 + CONTEXT_S) * sfreq))

        ch_idx  = raw.ch_names.index(ch)
        t_win   = times[i0:i1]
        raw_win = data[ch_idx, i0:i1] * 1e6          # V → µV
        flt_win = all_filtered[ch][i0:i1] * 1e6
        z_win   = all_z[ch][i0:i1]

        fig, axes = plt.subplots(3, 1, figsize=(11, 5), sharex=True)
        fig.suptitle(
            f'{SUBJECT}  |  {ch}  |  t = {t0:.2f} s  '
            f'dur = {row["duration_ms"]:.0f} ms  peak z = {row["peak_z"]:.1f}',
            fontsize=10
        )

        axes[0].plot(t_win, raw_win, lw=0.8, color='steelblue')
        axes[0].axvspan(t0, t1, color='red', alpha=0.15)
        axes[0].set_ylabel('Raw LFP (µV)')

        axes[1].plot(t_win, flt_win, lw=0.8, color='darkorange')
        axes[1].axvspan(t0, t1, color='red', alpha=0.15)
        axes[1].set_ylabel(f'{RIPPLE_LO}–{RIPPLE_HI} Hz (µV)')

        axes[2].plot(t_win, z_win, lw=0.8, color='seagreen')
        axes[2].axhline(Z_THRESHOLD, color='red', lw=0.8, ls='--',
                        label=f'z = {Z_THRESHOLD}')
        axes[2].axvspan(t0, t1, color='red', alpha=0.15)
        axes[2].set_ylabel('Z-score')
        axes[2].set_xlabel('Time (s)')
        axes[2].legend(fontsize=8, loc='upper right')

        plt.tight_layout()
        plt.show()

## 5. Summary Statistics

Event count and rate per hippocampal channel.  
Expected range for human hippocampal ripples: **0.3 – 3 events/min** during an active task  
(human iEEG literature; higher during sleep/rest).

In [ ]:
if df_events.empty:
    print('No events — nothing to plot. Try lowering Z_THRESHOLD.')
else:
    rec_duration_min = times[-1] / 60
    ch_counts = df_events.groupby('channel').size().sort_values(ascending=False)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].bar(ch_counts.index, ch_counts.values, color='steelblue')
    axes[0].set_title('Event count per channel')
    axes[0].set_xlabel('Channel')
    axes[0].set_ylabel('# Events')
    axes[0].tick_params(axis='x', rotation=45)

    axes[1].bar(ch_counts.index,
                ch_counts.values / rec_duration_min,
                color='darkorange')
    axes[1].set_title('Event rate (events / min)')
    axes[1].set_xlabel('Channel')
    axes[1].set_ylabel('Events / minute')
    axes[1].tick_params(axis='x', rotation=45)

    axes[2].hist(df_events['duration_ms'], bins=30, color='seagreen', edgecolor='white')
    axes[2].axvline(df_events['duration_ms'].mean(), color='red', lw=1.5, ls='--',
                    label=f'mean = {df_events["duration_ms"].mean():.0f} ms')
    axes[2].set_title('Event duration distribution')
    axes[2].set_xlabel('Duration (ms)')
    axes[2].set_ylabel('Count')
    axes[2].legend(fontsize=9)

    plt.suptitle(f'{SUBJECT} / {SESSION}  —  {RIPPLE_LO}–{RIPPLE_HI} Hz  '
                 f'z>{Z_THRESHOLD} (max={MAX_Z})  |  {rec_duration_min:.1f} min',
                 fontsize=10)
    plt.tight_layout()
    plt.show()

    print(f'Recording duration : {rec_duration_min:.1f} min')
    print(f'Total events       : {len(df_events)}')
    print(f'Mean duration      : {df_events["duration_ms"].mean():.1f} ± {df_events["duration_ms"].std():.1f} ms')
    print(f'Mean peak z-score  : {df_events["peak_z"].mean():.2f}')
    print(f'Overall event rate : {len(df_events) / rec_duration_min:.2f} events/min')

## 6. Cross-Channel Consensus

A per-channel event is a **candidate** — it could be a true SWR or a local artifact.
Requiring the same time window to be active in **≥ 2 channels simultaneously** is a
simple but powerful false-positive filter: a true SWR is a large field potential that
propagates across nearby contacts, while noise is usually channel-local.

**Algorithm:**
1. Build a binary activity matrix  from 
2. Sum across channels →  = how many channels are active at sample *t*
3. Find contiguous regions where 
4. Apply the same duration filter as Stage 3

 means *at least 2 channels must overlap* — change to 3 for stricter criteria.

In [ ]:
MIN_CONSENSUS = 1   # minimum channels that must overlap simultaneously

n_samples = data.shape[1]

# 1. Binary activity matrix: (n_channels x n_samples)
binary = np.zeros((len(raw.ch_names), n_samples), dtype=np.int8)
for i, ch in enumerate(raw.ch_names):
    for s, e in all_events[ch]:
        binary[i, s:e] = 1

# 2. Consensus signal: how many channels are active at each sample
consensus = binary.sum(axis=0)

# 3. Threshold crossings on consensus signal
above = consensus >= MIN_CONSENSUS
diff  = np.diff(above.astype(np.int8))
starts = np.where(diff ==  1)[0] + 1
ends   = np.where(diff == -1)[0] + 1
if above[0]:  starts = np.concatenate([[0],             starts])
if above[-1]: ends   = np.concatenate([ends, [len(above)]])

# 4. Duration filter (same limits as per-channel detection)
min_samp = int(MIN_DURATION * sfreq)
max_samp = int(MAX_DURATION * sfreq)
consensus_events = [(int(s), int(e)) for s, e in zip(starts, ends)
                    if min_samp <= (e - s) <= max_samp]

total_per_ch = len(df_events)
total_consensus = len(consensus_events)
reduction = 100 * (1 - total_consensus / total_per_ch) if total_per_ch else 0

print(f'MIN_CONSENSUS threshold : {MIN_CONSENSUS} channels')
print(f'Per-channel total       : {total_per_ch:>6,} events')
print(f'After consensus filter  : {total_consensus:>6,} events')
print(f'Reduction               : {reduction:.1f}%')

# ── Peak channel agreement per consensus event ──────────────────────────────
peak_counts = [int(consensus[s:e].max()) for s, e in consensus_events]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: histogram — how many channels agree at the peak of each event
axes[0].hist(peak_counts,
             bins=range(MIN_CONSENSUS, len(raw.ch_names) + 2),
             color='steelblue', edgecolor='white', align='left')
axes[0].set_title('Peak channel agreement per consensus event')
axes[0].set_xlabel('Number of channels simultaneously active')
axes[0].set_ylabel('Event count')
axes[0].set_xticks(range(MIN_CONSENSUS, len(raw.ch_names) + 1))

# Right: full-recording consensus signal
t_axis = np.arange(n_samples) / sfreq / 60   # minutes
axes[1].plot(t_axis, consensus, lw=0.4, color='steelblue', alpha=0.8)
axes[1].axhline(MIN_CONSENSUS, color='red', lw=1.2, ls='--',
                label=f'threshold = {MIN_CONSENSUS} channels')
axes[1].set_title('Channels active simultaneously (full recording)')
axes[1].set_xlabel('Time (min)')
axes[1].set_ylabel('N channels active')
axes[1].legend(fontsize=9)

plt.suptitle(
    f'{SUBJECT} / {SESSION}  —  Cross-channel consensus (≥{MIN_CONSENSUS} channels)  '
    f'→  {total_consensus:,} events  ({reduction:.0f}% reduction)',
    fontsize=10
)
plt.tight_layout()
plt.show()


## 7. Alignment with Behavioral Events — Gap Periods

**Goal:** check whether consensus SWR events cluster in the inter-word gaps,
where memory encoding/consolidation is expected.

**Synchronisation note:** the EDF started recording *before* the experiment began.
The TSV  column is relative to the session-start event, but
 is relative to EDF sample 0 — the same coordinate as .
We therefore use  throughout (same trick as DRAFT1 Step 5).

**Gap window:** from word-offset () to next-word onset.
A consensus SWR *counts* if any part of it overlaps the gap window.

In [ ]:
WORD_DURATION_S = 1.6   # fixed ISI presentation duration (s)

# ── 1. Load TSV ────────────────────────────────────────────────────────────
tsv_path = os.path.join(BIDS_ROOT, SUBJECT, SESSION, 'beh',
                        f'{SUBJECT}_{SESSION}_task-FR1_beh.tsv')
beh_df = pd.read_csv(tsv_path, sep='	')

# ── 2. WORD events from real lists (list > 0) ──────────────────────────────
# sample/500 → seconds from EDF start  (same coordinate as raw.times)
words_df = beh_df[(beh_df['trial_type'] == 'WORD') & (beh_df['list'] > 0)].copy()
words_df = words_df.sort_values(['list', 'serialpos']).reset_index(drop=True)
words_df['onset_edf_s'] = words_df['sample'] / SFREQ_EXPECTED

print(f'WORD events : {len(words_df)} across {words_df[list].nunique()} lists')

# ── 3. Build gap windows (word_i offset → word_{i+1} onset) ───────────────
gap_rows = []
for list_num, grp in words_df.groupby('list'):
    grp = grp.sort_values('serialpos').reset_index(drop=True)
    for i in range(len(grp) - 1):
        g_start = grp.loc[i,   'onset_edf_s'] + WORD_DURATION_S
        g_end   = grp.loc[i+1, 'onset_edf_s']
        if g_end > g_start:
            gap_rows.append({'list': list_num, 'gap_num': i + 1,
                             'start_s': g_start, 'end_s': g_end,
                             'duration_ms': (g_end - g_start) * 1000})

df_gaps = pd.DataFrame(gap_rows)
print(f'Gap windows : {len(df_gaps)}  |  '
      f'mean duration {df_gaps["duration_ms"].mean():.0f} ms  '
      f'(range {df_gaps["duration_ms"].min():.0f}–{df_gaps["duration_ms"].max():.0f} ms)')

# ── 4. Convert consensus events to seconds ─────────────────────────────────
consensus_s = [(s / sfreq, e / sfreq) for s, e in consensus_events]

# ── 5. Overlap test ────────────────────────────────────────────────────────
def _overlaps(s1, e1, s2, e2):
    return s1 < e2 and e1 > s2

in_gap = np.zeros(len(consensus_s), dtype=bool)
for idx, (sw_s, sw_e) in enumerate(consensus_s):
    for _, gap in df_gaps.iterrows():
        if _overlaps(sw_s, sw_e, gap['start_s'], gap['end_s']):
            in_gap[idx] = True
            break

# ── 6. Rates ───────────────────────────────────────────────────────────────
total_gap_s    = df_gaps['duration_ms'].sum() / 1000
total_rec_s    = float(times[-1])
nongap_s       = total_rec_s - total_gap_s

n_total  = len(consensus_s)
n_gap    = int(in_gap.sum())
n_nongap = n_total - n_gap

rate_gap    = n_gap    / (total_gap_s  / 60)
rate_nongap = n_nongap / (nongap_s     / 60)
enrichment  = rate_gap / rate_nongap if rate_nongap > 0 else float('nan')

print(f'Recording total      : {total_rec_s/60:.1f} min')
print(f'Gap time total       : {total_gap_s/60:.2f} min  ({100*total_gap_s/total_rec_s:.1f}% of recording)')
print(f'Consensus SWR total  : {n_total}')
print(f'SWR during gaps      : {n_gap}  ({100*n_gap/n_total:.1f}%)')
print(f'Rate during gaps     : {rate_gap:.2f} events/min')
print(f'Rate outside gaps    : {rate_nongap:.2f} events/min')
print(f'Enrichment ratio     : {enrichment:.2f}×  (>1 = SWR enriched in gaps)')

# ── 7. Plots ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: rate comparison bar chart
axes[0].bar(['During gaps', 'Outside gaps'],
            [rate_gap, rate_nongap],
            color=['steelblue', 'lightgray'], edgecolor='black', width=0.5)
axes[0].set_ylabel('SWR rate (events / min)')
axes[0].set_title(f'SWR rate: gap vs. non-gap(enrichment = {enrichment:.2f}×)')
for spine in ['top', 'right']:
    axes[0].spines[spine].set_visible(False)

# Right: timeline scatter coloured by gap/non-gap
swr_t_min = np.array([s for s, e in consensus_s]) / 60
axes[1].scatter(swr_t_min[~in_gap], np.ones((~in_gap).sum()),
                color='lightgray', s=6, alpha=0.5, label=f'Non-gap ({n_nongap})')
axes[1].scatter(swr_t_min[in_gap],  np.ones(in_gap.sum()),
                color='steelblue',  s=12, alpha=0.9, label=f'Gap SWR ({n_gap})')
for _, gap in df_gaps.iterrows():
    axes[1].axvspan(gap['start_s'] / 60, gap['end_s'] / 60,
                    alpha=0.10, color='steelblue', lw=0)
axes[1].set_xlabel('Time (min)')
axes[1].set_yticks([])
axes[1].set_title('SWR events on recording timeline')
axes[1].legend(fontsize=9)

plt.suptitle(
    f'{SUBJECT} / {SESSION}  —  SWR × inter-word gap alignment  '
    f'({n_gap}/{n_total} SWR in gaps)',
    fontsize=10
)
plt.tight_layout()
plt.show()


## 8. Gap SWR — Visual Inspection

Same 3-panel layout as Cell 4, applied to the events that overlapped an inter-word gap.
For each consensus event the channel with the **highest peak z-score** is shown.
The title includes which list and gap number the event fell into.

In [ ]:
CONTEXT_S = 0.200   # s — context window before/after event (same as cell 4)

gap_event_indices = np.where(in_gap)[0]

if len(gap_event_indices) == 0:
    print('No gap SWR events found.')
else:
    print(f'Showing all {len(gap_event_indices)} gap SWR events:\n')

    for ev_idx in gap_event_indices:
        sw_s, sw_e = consensus_s[ev_idx]
        s_samp = int(sw_s * sfreq)
        e_samp = int(sw_e * sfreq)

        # Channel with highest peak z during this event window
        best_ch = max(
            raw.ch_names,
            key=lambda ch: float(all_z[ch][s_samp:e_samp].max())
        )

        # Context window indices
        i0 = max(0, int((sw_s - CONTEXT_S) * sfreq))
        i1 = min(len(times) - 1, int((sw_e + CONTEXT_S) * sfreq))

        ch_idx  = raw.ch_names.index(best_ch)
        t_win   = times[i0:i1]
        raw_win = data[ch_idx, i0:i1] * 1e6
        flt_win = all_filtered[best_ch][i0:i1] * 1e6
        z_win   = all_z[best_ch][i0:i1]
        peak_z  = float(all_z[best_ch][s_samp:e_samp].max())
        dur_ms  = (sw_e - sw_s) * 1000

        # Which gap window does this event belong to?
        match = df_gaps[df_gaps.apply(
            lambda r: sw_s < r['end_s'] and sw_e > r['start_s'], axis=1
        )]
        gap_label = (f'List {int(match.iloc[0]["list"])}, gap after word {int(match.iloc[0]["gap_num"])}'
                     if len(match) else 'gap ?')

        fig, axes = plt.subplots(3, 1, figsize=(11, 5), sharex=True)
        fig.suptitle(
            f'{SUBJECT}  |  {best_ch}  |  t = {sw_s:.2f} s  '
            f'dur = {dur_ms:.0f} ms  peak z = {peak_z:.1f}  [{gap_label}]',
            fontsize=10
        )

        axes[0].plot(t_win, raw_win, lw=0.8, color='steelblue')
        axes[0].axvspan(sw_s, sw_e, color='red', alpha=0.15)
        axes[0].set_ylabel('Raw LFP (µV)')

        axes[1].plot(t_win, flt_win, lw=0.8, color='darkorange')
        axes[1].axvspan(sw_s, sw_e, color='red', alpha=0.15)
        axes[1].set_ylabel(f'{RIPPLE_LO}–{RIPPLE_HI} Hz (µV)')

        axes[2].plot(t_win, z_win, lw=0.8, color='seagreen')
        axes[2].axhline(Z_THRESHOLD, color='red', lw=0.8, ls='--',
                        label=f'z = {Z_THRESHOLD}')
        axes[2].axvspan(sw_s, sw_e, color='red', alpha=0.15)
        axes[2].set_ylabel('Z-score')
        axes[2].set_xlabel('Time (s)')
        axes[2].legend(fontsize=8, loc='upper right')

        plt.tight_layout()
        plt.show()

## Troubleshooting

| Problem | Fix |
|---------|-----|
| No events detected | Lower `Z_THRESHOLD` to 2.5 |
| Events look like noise | Raise `Z_THRESHOLD` to 3.5; check notch filter |
| z=46 events still appearing | Lower `MAX_Z` to 10 |
| Channel not found in EDF | The `[warn]` line in Cell 2 shows EDF channel names for debugging |
| LDA2 / other channel missing | Check if it appears in `bad_hippo` list printed in Cell 1 |